# MBPP Quickstart: Text-to-Code Generation & Search

This notebook gives everyone an end-to-end working baseline for both MBPP tasks:

| Task | What it does | Model | Key Metric |
|------|-------------|-------|------------|
| **Task 1: Code Generation** | Given a text description, *generate* Python code | Qwen2.5-Coder 1.5B Instruct (QLoRA) | pass@1 |
| **Task 2: Code Search** | Given a text description, *retrieve* the best matching code | UniXcoder-base 125M | MRR |

**Dataset:** [MBPP](https://huggingface.co/datasets/google-research-datasets/mbpp) — 1k crowd-sourced Python problems with unit tests.

**Runtime:** ~25-35 min on Colab T4 GPU (fine-tuning + evaluation).

---
## 0a. Install Dependencies (run once, then runtime auto-restarts)

This cell installs packages and **restarts the runtime** so they take effect.
After restart, skip this cell and continue from **0b** below.

In [ ]:
!pip install -q datasets "transformers==4.44.2" accelerate peft sentence-transformers sentencepiece bitsandbytes "trl==0.9.6"

# Auto-restart runtime so the new transformers version is loaded
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

---
## 0b. Imports & GPU Check

**Start here** after the runtime restarts (or on subsequent runs).

In [ ]:
import transformers
print(f"transformers version: {transformers.__version__}")  # should be 4.44.2

import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
)
from peft import LoraConfig
from trl import SFTTrainer
import bitsandbytes  # verify install
from IPython.display import display, HTML

# Check GPU
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    device = torch.device("cpu")
    print("No GPU found — will use CPU (slower). Go to Runtime > Change runtime type > T4 GPU.")

---
## 1. Load & Explore the MBPP Dataset

MBPP = **Mostly Basic Python Problems**. Each example has:
- `text`: Natural language description of what the function should do
- `code`: A reference Python solution
- `test_list`: Unit test assertions to verify correctness

This is the raw material for *both* tasks.

In [ ]:
dataset = load_dataset("google-research-datasets/mbpp")
print(dataset)
print(f"\nSplits: train={len(dataset['train'])}, test={len(dataset['test'])}, "
      f"validation={len(dataset['validation'])}, prompt={len(dataset['prompt'])}")

In [ ]:
# Look at one example in detail
example = dataset["test"][0]
print("=" * 60)
print(f"Task ID: {example['task_id']}")
print(f"\n--- Description (model input) ---")
print(example["text"])
print(f"\n--- Reference Code (ground truth) ---")
print(example["code"])
print(f"\n--- Unit Tests (how we check correctness) ---")
for t in example["test_list"]:
    print(f"  {t}")

In [ ]:
# Quick stats to understand the data
test_set = dataset["test"]

desc_lengths = [len(ex["text"].split()) for ex in test_set]
code_lengths = [len(ex["code"].split("\n")) for ex in test_set]

print(f"Test set: {len(test_set)} examples")
print(f"\nDescription length (words): min={min(desc_lengths)}, "
      f"median={sorted(desc_lengths)[len(desc_lengths)//2]}, max={max(desc_lengths)}")
print(f"Code length (lines):        min={min(code_lengths)}, "
      f"median={sorted(code_lengths)[len(code_lengths)//2]}, max={max(code_lengths)}")

# Show a few more examples as a quick scan
print("\n" + "=" * 60)
print("Sample descriptions from the test set:")
for i in [0, 100, 200, 300, 400]:
    print(f"  [{i:3d}] {test_set[i]['text'][:100]}...")

---
## 2. Task 1 — Code Generation with Qwen2.5-Coder (QLoRA)

**Goal:** Given a natural language description, generate working Python code.

**Model:** [**Qwen2.5-Coder-1.5B-Instruct**](https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B-Instruct) (1.5B params, decoder-only, instruct-tuned)

**Architecture:** Decoder-only causal language model with a chat template. Unlike encoder-decoder models (T5), the entire input + output flows through a single transformer stack. The instruct tuning means it already understands instructions, so it can generate code zero-shot.

**Plan:**
1. **Zero-shot baseline** — evaluate the model out of the box (no training)
2. **QLoRA fine-tune** — adapt the model on MBPP training data
3. **Re-evaluate** — measure improvement from fine-tuning

**What is QLoRA?**
- The base model is loaded in **4-bit quantization** (NF4) — reduces 1.5B params from ~3GB to ~500MB VRAM
- Base weights are **frozen** (not trained)
- Small **LoRA adapter matrices** (~2M trainable params) are injected into attention layers
- Only the adapters are trained → fast, memory-efficient fine-tuning on a free T4 GPU

In [ ]:
# Load Qwen2.5-Coder-1.5B-Instruct in 4-bit quantization
GEN_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
gen_tokenizer.pad_token = gen_tokenizer.eos_token

gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

total_params = sum(p.numel() for p in gen_model.parameters())
print(f"Model: {GEN_MODEL}")
print(f"Parameters: {total_params / 1e9:.1f}B (loaded in 4-bit)")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
def generate_code(description: str, max_new_tokens: int = 512) -> str:
    """Generate Python code from a natural language description using Qwen2.5-Coder.

    Uses the model's chat template: system prompt tells it to output code only,
    user message is the task description. The model generates autoregressively.
    """
    messages = [
        {"role": "system", "content": "Write a Python function that solves the given task. Only output the code, no explanations."},
        {"role": "user", "content": description},
    ]
    text = gen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = gen_tokenizer([text], return_tensors="pt").to(gen_model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=gen_tokenizer.eos_token_id,
        )

    response = gen_tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    response = response.strip()

    # Strip markdown code fences (```python ... ```) that instruct models often add
    if response.startswith("```"):
        response = response.split("\n", 1)[1] if "\n" in response else response[3:]
    if response.endswith("```"):
        response = response[:-3]

    return response.strip()


# Quick test — zero-shot (no fine-tuning yet)
example = test_set[0]
generated = generate_code(example["text"])

print("--- Input Description ---")
print(example["text"])
print("\n--- Generated Code (ZERO-SHOT, no fine-tuning) ---")
print(generated)
print("\n--- Reference Code ---")
print(example["code"])
print()
print("^ The instruct model can already generate code! But it may not match")
print("  the exact function signatures MBPP tests expect. Fine-tuning helps.")

### Evaluating with pass@1

**pass@1** = "Does the generated code pass all unit tests on the first try?"

This is the standard metric for code generation. We:
1. Execute the generated code
2. Run the test assertions against it
3. If all assertions pass → success (1), otherwise → fail (0)
4. Average across all examples

In [ ]:
import signal
from contextlib import contextmanager


@contextmanager
def time_limit(seconds: int):
    """Timeout guard — generated code might hang (infinite loops etc.)."""
    def handler(signum, frame):
        raise TimeoutError()
    old = signal.signal(signal.SIGALRM, handler)
    signal.alarm(seconds)
    try:
        yield
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old)


def check_correctness(generated_code: str, test_list: list, timeout: int = 5) -> bool:
    """Execute generated code and run test assertions. Returns True if all tests pass."""
    try:
        with time_limit(timeout):
            exec_globals = {}
            exec(generated_code, exec_globals)
            for test in test_list:
                exec(test, exec_globals)
        return True
    except Exception:
        return False


# Quick sanity check: does the *reference* code pass its own tests?
ref_pass = check_correctness(test_set[0]["code"], test_set[0]["test_list"])
print(f"Reference code passes its own tests: {ref_pass}")

# Check our generated code
gen_pass = check_correctness(generated, test_set[0]["test_list"])
print(f"Generated code passes tests: {gen_pass}")

In [ ]:
# Zero-shot evaluation on N_EVAL examples (before any fine-tuning)
N_EVAL = 50

zs_results = []
for i in range(N_EVAL):
    ex = test_set[i]
    gen_code = generate_code(ex["text"])
    passed = check_correctness(gen_code, ex["test_list"])
    zs_results.append(passed)
    if (i + 1) % 10 == 0:
        running_rate = sum(zs_results) / len(zs_results)
        print(f"  [{i+1:3d}/{N_EVAL}] running pass@1 = {running_rate:.1%}")

pass_at_1_zs = sum(zs_results) / len(zs_results)
print(f"\n{'=' * 40}")
print(f"Zero-shot pass@1 = {pass_at_1_zs:.1%} ({sum(zs_results)}/{len(zs_results)})")
print(f"{'=' * 40}")
print("The instruct model gets some right out of the box. Let's improve with fine-tuning.")

### QLoRA Fine-Tuning on MBPP

Now we fine-tune the model on MBPP's (description → code) pairs using **QLoRA**:

**How QLoRA works:**
- The base model stays **quantized to 4-bit** and **frozen** (no gradient updates)
- We inject small **LoRA adapter matrices** into the attention layers (`q_proj`, `v_proj`)
- Only these adapters (~2M params out of 1.5B) are trained
- This means we train **<0.2%** of the model's parameters!

**Training setup:**
- **Data:** 374 train examples formatted as chat conversations (system + user + assistant)
- **Trainer:** `SFTTrainer` from `trl` — handles chat formatting and packing automatically
- **Epochs:** 3, **LR:** 2e-4, **Batch size:** 4 (with gradient accumulation = 2 → effective batch 8)
- **Time:** ~15 min on T4

In [ ]:
# LoRA configuration — which layers to adapt and how
lora_config = LoraConfig(
    r=8,                          # rank of the adapter matrices (low = fewer params)
    lora_alpha=16,                # scaling factor (alpha/r = effective LR multiplier)
    target_modules=["q_proj", "v_proj"],  # inject adapters into attention Q and V projections
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Format training data as chat conversations for the instruct model
def format_example(example):
    messages = [
        {"role": "system", "content": "Write a Python function that solves the given task. Only output the code, no explanations."},
        {"role": "user", "content": example["text"]},
        {"role": "assistant", "content": example["code"]},
    ]
    return {"text": gen_tokenizer.apply_chat_template(messages, tokenize=False)}

train_dataset = dataset["train"].map(format_example)
val_dataset = dataset["validation"].map(format_example)

print(f"Training examples: {len(train_dataset)}")
print(f"Validation examples: {len(val_dataset)}")
print(f"\nSample formatted input (first 300 chars):")
print(train_dataset[0]["text"][:300])

In [ ]:
training_args = TrainingArguments(
    output_dir="./qwen-coder-mbpp-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    bf16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=gen_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=lora_config,
)

# Show trainable vs total params
trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
total = sum(p.numel() for p in trainer.model.parameters())
print(f"Trainable params: {trainable / 1e6:.1f}M / {total / 1e6:.0f}M ({100 * trainable / total:.2f}%)")
print("\nStarting QLoRA fine-tuning (3 epochs, ~15 min on T4)...")
trainer.train()

In [ ]:
# Evaluate pass@1 after QLoRA fine-tuning (same 50 examples as zero-shot)
gen_model.eval()

ft_results = []
for i in range(N_EVAL):
    ex = test_set[i]
    gen_code = generate_code(ex["text"])
    passed = check_correctness(gen_code, ex["test_list"])
    ft_results.append(passed)
    if (i + 1) % 10 == 0:
        running_rate = sum(ft_results) / len(ft_results)
        print(f"  [{i+1:3d}/{N_EVAL}] running pass@1 = {running_rate:.1%}")

pass_at_1_ft = sum(ft_results) / len(ft_results)

print(f"\n{'=' * 50}")
print(f"Task 1 Code Generation — Zero-shot vs Fine-tuned")
print(f"{'=' * 50}")
print(f"  Zero-shot pass@1:  {pass_at_1_zs:.1%} ({sum(zs_results)}/{N_EVAL})")
print(f"  Fine-tuned pass@1: {pass_at_1_ft:.1%} ({sum(ft_results)}/{N_EVAL})")
print(f"  Delta:             {pass_at_1_ft - pass_at_1_zs:+.1%}")
print(f"{'=' * 50}")

# Show a few examples
for i in [0, 5, 10]:
    ex = test_set[i]
    gen_code = generate_code(ex["text"])
    passed = check_correctness(gen_code, ex["test_list"])
    status = "PASS" if passed else "FAIL"
    print(f"\n--- Example {i} [{status}] ---")
    print(f"Description: {ex['text'][:100]}")
    print(f"Generated:\n{gen_code[:200]}")

---
## 3. Task 2 — Text-to-Code Search (Embedding Baseline)

**Goal:** Given a text description, *find* the best matching code from a pool of candidates.

**Key difference from Task 1:** We're *retrieving* existing code, not *generating* new code.

**Model:** [**UniXcoder-base**](https://huggingface.co/microsoft/unixcoder-base) (125M params, RoBERTa-based)
- Pre-trained on both programming languages and natural language
- Produces 768-dim embeddings via CLS token pooling
- Works natively with HuggingFace `AutoModel` — no custom code needed

**Approach:**
1. Encode both text descriptions and code solutions into vectors using UniXcoder
2. Compute cosine similarity between text query and all code vectors
3. Rank by similarity — the closest code is our prediction

Think of it like a search engine: the text description is your query, the code solutions are the documents.

In [ ]:
from transformers import AutoModel

# Load the embedding model (separate from the generation model)
EMB_MODEL = "microsoft/unixcoder-base"

emb_tokenizer = AutoTokenizer.from_pretrained(EMB_MODEL)
emb_model = AutoModel.from_pretrained(EMB_MODEL).to(device)

print(f"Model: {EMB_MODEL}")
print(f"Parameters: {sum(p.numel() for p in emb_model.parameters()) / 1e6:.0f}M")
print(f"Embedding dimension: {emb_model.config.hidden_size}")

In [ ]:
@torch.no_grad()
def encode_batch(texts: list, model, batch_size: int = 32) -> np.ndarray:
    """Encode a list of strings into normalized embedding vectors using CLS pooling."""
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        inputs = emb_tokenizer(
            batch, return_tensors="pt", max_length=512, truncation=True, padding=True
        ).to(device)
        outputs = model(**inputs)
        # CLS token embedding (first token of last hidden state)
        embeddings = outputs.last_hidden_state[:, 0, :]
        # Normalize so cosine similarity = dot product
        embeddings = torch.nn.functional.normalize(embeddings, dim=-1)
        all_embeddings.append(embeddings.cpu().numpy())
    return np.concatenate(all_embeddings, axis=0)


# Encode all text descriptions and code solutions from the test set
descriptions = [ex["text"] for ex in test_set]
codes = [ex["code"] for ex in test_set]

print(f"Encoding {len(descriptions)} descriptions...")
query_embeddings = encode_batch(descriptions, emb_model)

print(f"Encoding {len(codes)} code solutions...")
code_embeddings = encode_batch(codes, emb_model)

print(f"\nQuery embeddings shape: {query_embeddings.shape}")
print(f"Code embeddings shape:  {code_embeddings.shape}")

In [ ]:
# Compute similarity matrix: each query vs. all codes
# Since embeddings are normalized, dot product = cosine similarity
similarity_matrix = query_embeddings @ code_embeddings.T
print(f"Similarity matrix shape: {similarity_matrix.shape}")
print(f"(Each row = one query's similarity to all {len(codes)} code solutions)")

# For each query, rank all codes by similarity (descending)
rankings = np.argsort(-similarity_matrix, axis=1)

# Show an example: where does the correct code rank for query 0?
query_idx = 0
correct_code_idx = query_idx  # In MBPP, query i matches code i
rank = np.where(rankings[query_idx] == correct_code_idx)[0][0] + 1

print(f"\nExample: Query 0 = \"{descriptions[0][:80]}...\"")
print(f"  Correct code ranked at position: {rank} out of {len(codes)}")
print(f"  Top-5 retrieved code indices: {rankings[query_idx][:5].tolist()}")

### Evaluating with MRR and Recall@K

- **MRR (Mean Reciprocal Rank):** Average of `1/rank` for the correct result. MRR=1.0 means always rank 1.
- **Recall@K:** What fraction of queries have the correct code in the top K results?

In [ ]:
def compute_search_metrics(rankings: np.ndarray, n: int) -> dict:
    """Compute MRR and Recall@K. Assumes query i should match code i."""
    reciprocal_ranks = []
    recall_at = {1: 0, 5: 0, 10: 0}

    for i in range(n):
        rank = np.where(rankings[i] == i)[0][0] + 1  # 1-indexed
        reciprocal_ranks.append(1.0 / rank)
        for k in recall_at:
            if rank <= k:
                recall_at[k] += 1

    mrr = np.mean(reciprocal_ranks)
    recall_at = {k: v / n for k, v in recall_at.items()}
    return {"MRR": mrr, **{f"Recall@{k}": v for k, v in recall_at.items()}}


metrics = compute_search_metrics(rankings, len(test_set))

print("=" * 40)
print("Task 2 Text-to-Code Search Results")
print("=" * 40)
for name, value in metrics.items():
    print(f"  {name:12s} = {value:.3f}")

In [ ]:
# Visualize: show top-5 retrieved codes for a few queries
for query_idx in [0, 50, 200]:
    print("=" * 60)
    print(f"Query [{query_idx}]: {descriptions[query_idx][:100]}")
    correct_rank = np.where(rankings[query_idx] == query_idx)[0][0] + 1
    print(f"Correct code rank: {correct_rank}")
    print()
    for j, code_idx in enumerate(rankings[query_idx][:5]):
        sim = similarity_matrix[query_idx, code_idx]
        tag = " <-- CORRECT" if code_idx == query_idx else ""
        print(f"  Rank {j+1} (sim={sim:.3f}){tag}:")
        # Show first 2 lines of code
        code_preview = "\n".join(codes[code_idx].split("\n")[:2])
        print(f"    {code_preview}")
    print()

---
## 4. Task 2 — Fine-Tuned Search (Optional Extension)

The pre-trained embeddings give a baseline, but we can improve by fine-tuning with **contrastive learning**:
- Positive pair: (description, its matching code)
- Negative pairs: (description, non-matching codes from the same batch)
- The model learns to push matching pairs closer and non-matching pairs apart.

> **Skip this section** if you want to proceed to the summary first.

In [ ]:
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

# Load a fresh copy of the embedding model for fine-tuning (so we can compare before/after)
ft_emb_model = AutoModel.from_pretrained(EMB_MODEL).to(device)
ft_emb_model.train()

# Simple dataset: pairs of (description, code)
class TextCodeDataset(Dataset):
    def __init__(self, hf_dataset):
        self.texts = [ex["text"] for ex in hf_dataset]
        self.codes = [ex["code"] for ex in hf_dataset]
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.codes[idx]

train_ds = TextCodeDataset(dataset["train"])
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)

# Multiple Negatives Ranking Loss (in-batch negatives):
# For each (text_i, code_i) pair in the batch, all other codes are negatives.
# Loss = -log( exp(sim(text_i, code_i)) / sum_j exp(sim(text_i, code_j)) )
def mnrl_loss(text_emb, code_emb, temperature=0.05):
    """In-batch contrastive loss. text_emb and code_emb are (B, D) normalized."""
    scores = text_emb @ code_emb.T / temperature  # (B, B)
    labels = torch.arange(scores.size(0), device=scores.device)
    return F.cross_entropy(scores, labels)

optimizer = torch.optim.AdamW(ft_emb_model.parameters(), lr=2e-5, weight_decay=0.01)

print(f"Training pairs: {len(train_ds)}")
print(f"Batches per epoch: {len(train_loader)}")
print("\nStarting contrastive fine-tuning (3 epochs, ~5 min on T4)...")

for epoch in range(3):
    total_loss = 0
    for batch_texts, batch_codes in train_loader:
        # Encode text descriptions — CLS pooling
        text_inputs = emb_tokenizer(
            list(batch_texts), return_tensors="pt", max_length=512, truncation=True, padding=True
        ).to(device)
        text_emb = ft_emb_model(**text_inputs).last_hidden_state[:, 0, :]
        text_emb = F.normalize(text_emb, dim=-1)

        # Encode code solutions — CLS pooling
        code_inputs = emb_tokenizer(
            list(batch_codes), return_tensors="pt", max_length=512, truncation=True, padding=True
        ).to(device)
        code_emb = ft_emb_model(**code_inputs).last_hidden_state[:, 0, :]
        code_emb = F.normalize(code_emb, dim=-1)

        loss = mnrl_loss(text_emb, code_emb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"  Epoch {epoch + 1}/3 — avg loss: {avg_loss:.4f}")

print("Done!")

In [ ]:
# Re-encode with the fine-tuned model and compare metrics
ft_emb_model.eval()

print("Re-encoding with fine-tuned model...")
query_emb_ft = encode_batch(descriptions, ft_emb_model)
code_emb_ft = encode_batch(codes, ft_emb_model)

sim_matrix_ft = query_emb_ft @ code_emb_ft.T
rankings_ft = np.argsort(-sim_matrix_ft, axis=1)

metrics_ft = compute_search_metrics(rankings_ft, len(test_set))

print("\nTask 2 Search — Before vs After Fine-Tuning")
print("=" * 50)
print(f"{'Metric':12s} {'Pre-trained':>12s} {'Fine-tuned':>12s} {'Delta':>10s}")
print("-" * 50)
for name in metrics:
    before = metrics[name]
    after = metrics_ft[name]
    print(f"{name:12s} {before:12.3f} {after:12.3f} {after - before:+10.3f}")

---
## 5. Results Summary & Next Steps

### Baseline Results

In [ ]:
print("MBPP Quickstart — Results Summary")
print("=" * 60)
print(f"{'Task':<35s} {'Metric':<12s} {'Score':>8s}")
print("-" * 60)
print(f"{'Task 1: Generation (zero-shot)':<35s} {'pass@1':<12s} {pass_at_1_zs:>8.1%}")
print(f"{'Task 1: Generation (QLoRA fine-tuned)':<35s} {'pass@1':<12s} {pass_at_1_ft:>8.1%}")
for name, value in metrics.items():
    print(f"{'Task 2: Search (pre-trained)':<35s} {name:<12s} {value:>8.1%}")
if 'metrics_ft' in dir():
    for name, value in metrics_ft.items():
        print(f"{'Task 2: Search (fine-tuned)':<35s} {name:<12s} {value:>8.1%}")
print("=" * 60)

### How This Maps to the Intermediate Update

| IU Section | What to put there |
|---|---|
| **Task Statement** | Two tasks on MBPP: (1) generate code from NL, (2) retrieve code from NL |
| **Proposed Method** | Qwen2.5-Coder 1.5B Instruct + QLoRA for generation (decoder-only, 4-bit quantized, LoRA fine-tuned), UniXcoder-base 125M for search (CLS embedding + contrastive learning) |
| **Progress** | Working baselines with pass@1 (zero-shot and fine-tuned) and MRR numbers from this notebook |
| **Proposed Evaluation** | pass@k for generation, MRR/Recall@k for search — both demonstrated here |
| **Resources** | MBPP dataset (HuggingFace), Google Colab (T4 GPU), HuggingFace Transformers, PEFT, TRL |

### Ideas for Improvement (for the final project)

**Task 1 — Generation:**
- Try larger models: Qwen2.5-Coder-7B, StarCoder2, DeepSeek-Coder with QLoRA
- Higher LoRA rank (r=16, r=32) or more target modules (k_proj, o_proj, gate_proj, etc.)
- Prompt engineering: include example I/O from test cases in the prompt
- pass@k with k>1: sample multiple solutions and pick the best

**Task 2 — Search:**
- Try different embedding models: CodeBERT, GraphCodeBERT, CodeT5+ (with newer transformers)
- Hard negative mining: train on difficult negatives, not just random
- Hybrid retrieval: combine embedding similarity with BM25 lexical matching
- Cross-encoder re-ranking: use a second model to re-rank top candidates